In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# create our numpy random number generator
rng = np.random.default_rng(42)

# Statistical Inference Based on Bayesian Updating

Classical statistical inference answers the question: how likely would I be to see a result as extreme as the data I actually saw, given some assumed null hypothesis?

Bayesian inference asks a different question: how should I update my initial beliefs about a data generating process, given one set of outputs I've observed of that process?

We'll again use code that simulates distributions (data generating processes) to illustrate Bayesian inference, with our reaction times dataset:
- We will start with a "uniform" prior belief: reaction time after a beer increases, on average, by some value in a range, but I have no idea where within that range it is.
- For each possible value of that parameter (the average reaction time increase), there's some likelihood of producing the outcome we saw.
- Those likelihoods collectively determine our posterior beliefs about the plausibility of those parameter values.


In [ ]:
exp_results = pd.read_csv('data/reaction_times.csv')

In [ ]:
exp_results['increase'] = exp_results['after'] - exp_results['before']

In [ ]:
exp_results.head()

In [ ]:
exp_results['increase'].describe()

# Uniform Prior
I'm going to make a parametric assumption about how a person's change in reaction is generated
- A person's change in reaction time is drawn from a normal distribution, which has two parameters, a mean and standard deviation:
  - The *mean* is somewhere between -0.05 seconds and +0.20 seconds, with all values in that range equally plausible to me (before I see the data). My reasoning for starting with these beliefs about the mean change in reaction time is based on some past literature or personal experiences:
    - It's possible that people actually get a little faster from Before to After, because of the practice effect. Perhaps that could be as much as 0.05 seconds.
    - But beer makes people slower, maybe by as much 0.2 seconds
  - The *standard deviation* is 0.266251, which is empirically calculated from my (fake) experiment results. 

We will operationalize our uncertainty about the mean change in reaction time as a "grid" of possible values for the mean, and then try to make inferences about which of those possible values is more plausible, given the data we saw in the experiment.

In [ ]:
actual_std = exp_results['increase'].std()
actual_std

In [ ]:
# grid of possible means from -0.05 to +0.2
possible_means = np.linspace(-0.05, 0.2, 26)
possible_means

## Simulate Sampling Distribution for Each Possible Value of Mean

For each possible value of the mean, we will simulate 1000 40-person samples and compute their means. In other words, for each possible value, we simulate its sampling distribution.

Remember this useful function from previous notebooks

In [ ]:
# simulate the distribution of some sample statistic, given a simulator for the underlying distribution
def simulate_sampling_distribution(
    underlying_distribution_simulator, sample_statistic, n_simulations=1000, sample_size=100
):
    samples = [underlying_distribution_simulator(sample_size) for _ in range(n_simulations)]
    return pd.DataFrame(
        {
            "statistic": [sample_statistic(sample) for sample in samples],
        }
    )

In [ ]:
n_simulations = 1000

In [ ]:
sampling_distributions = pd.DataFrame({
    'mean_change_in_reaction_time': possible_means,
    'sampling_distribution': [simulate_sampling_distribution(
            lambda n: rng.normal(param_value, actual_std, n),
            lambda x: x.mean(),
            n_simulations=n_simulations,
            sample_size=40) for param_value in possible_means]
})


In [ ]:
sampling_distributions.head()

In [ ]:
sampling_distributions.loc[1, 'sampling_distribution'].head()

## Compute likelihood that each produces sample mean that we saw

We saw a sample mean of 0.063628. 

For each of our possible_means, some of their 1000 sample means will be in the range [0.06, 0.07], where our observed sample mean is.
- Let's count how many.

In [ ]:
# for each sampling distribution, calculate how many of its values are in the desired range
sampling_distributions['likelihood_count'] = sampling_distributions['sampling_distribution'].apply(
    lambda df: ((df['statistic'] >= 0.06) & (df['statistic'] < 0.07)).sum())

We get a sample mean in the range [0.06, .07] more often when each person's reaction time change is around 0.06 than when each person's reaction time change is 0, or 0.15.
- remember that these counts are out of 1000 samples for each possible mean change in reaction time.

In [ ]:
sampling_distributions

## Convert those Likelihoods to Posterior Probabilities

- We started not having any reason to think that a mean change in reaction time of 0 was more or less likely than 0.4
- But out our samples with underlying mean change of 0, only 21 were in the range we actually observed
- Out of our samples with underlying mean change of 0.04, 67 were in that range
- So our posterior belief is that 0.04 is more than three times likely to be the true underlying parameter value than 0 is.
    - Of all the times we get a statistic in the target range, more than three times as many come from simulations with parameter 0.04 than from simulations with parameter value 0.

In [ ]:
# normalize likelihood_count column to get probabilities
sampling_distributions['posterior_belief'] = sampling_distributions['likelihood_count'] / sampling_distributions['likelihood_count'].sum()
sampling_distributions

In [ ]:
# plot the posterior belief as a histogram, with mean_change_in_reaction_time on the x-axis
plt.figure(figsize=(12, 6))
ax = sns.barplot(x='mean_change_in_reaction_time', y='posterior_belief', data=sampling_distributions)

# Make it display prettier...
# Get all the current x-axis tick positions
tick_positions = ax.get_xticks()

# Set every fifth label visible, hiding the rest
new_tick_labels = [f"{float(label.get_text()):.2f}" if index % 5 == 0 else '' for index, label in enumerate(ax.get_xticklabels())]

# Set the tick positions and labels
ax.set_xticks(tick_positions)  # Explicitly set tick positions
ax.set_xticklabels(new_tick_labels);  # ; suppresses some unwanted text output

## Compute "Credible Intervals" for the posterior belief

A Bayesian 95%-credible interval covers 95% of the posterior belief distribution. 
- It's analogous to a 95%-confidence interval in classical statistical inference.
- It means that you believe, given the data you've seen, there's a 95% probability that the true underlying mean lies in that range.
- If 0 lies outside that interval, you think there's a less than 5% chance that the underlying mean is 0 or less.

Thus, the meaning of Bayesian credible intervals is quite intuitive. 
- (They mean what one might wish classical confidence intervals would mean; but the intuitive interpretation for classical confidence intervals is not correct.)


In [ ]:
cdf = pd.DataFrame({
    'mean_change_in_reaction_time': possible_means,
    'cumulative_posterior_belief': sampling_distributions['posterior_belief'].cumsum()
})
cdf.set_index('mean_change_in_reaction_time', inplace=True)
cdf

There is more than 6% chance that the true mean change in reaction time is 0 or less, given the data we've seen.

In [ ]:
# find the range of values covering the middle 95% of the posterior belief
lower_bound = cdf.loc[cdf['cumulative_posterior_belief'] < 0.025].index.max()
upper_bound = cdf.loc[cdf['cumulative_posterior_belief'] > 0.975].index.min()

lower_bound, upper_bound

In [ ]:
# plot the posterior belief as a histogram, with mean_change_in_reaction_time on the x-axis
plt.figure(figsize=(12, 6))
ax = sns.barplot(x='mean_change_in_reaction_time', y='posterior_belief', data=sampling_distributions)

# Make it display prettier...
# Get all the current x-axis tick positions
tick_positions = ax.get_xticks()

# Set every fifth label visible, hiding the rest
new_tick_labels = [f"{float(label.get_text()):.2f}" if index % 5 == 0 else '' for index, label in enumerate(ax.get_xticklabels())]

# Set the tick positions and labels
ax.set_xticks(tick_positions)  # Explicitly set tick positions
ax.set_xticklabels(new_tick_labels);  # ; suppresses some unwanted text output

# add red vertical lines at lower_bound and upper_bound
ax.axvline(cdf.index.get_loc(lower_bound), color='red', linestyle='--')
ax.axvline(cdf.index.get_loc(upper_bound), color='red', linestyle='--')

Given the evidence from the experiment, our posterior belief is:
- There is a 95% chance that the true mean change in reaction time is between (-0.03, +0.15)
- So, we're not confident, beyond a reasonable doubt (5%) that the mean change in reaction time is greater than 0.
    - Similar conclusion to what we found with classical hypothesis testing
    - But with the Bayesian approach we can quantify our beliefs about the plausibility of different possible underlying mean change in reaction time, including how big that change might be.